In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Descarga de CEAS
Del CATDISS se descarga el catalogo oficial de CEAS. Como no se dispone de un csv de datos, se inspecciona el tráfico
de red y se descarga directamente con la api mediante la librería requests, dejandolo en la ruta de inputs para que esté
disponible para el tratamiento de la dimensión de servicios.

In [2]:
url = "https://servicios.jcyl.es/resobackend/services/recursos/mapa?tiposRecurso=D"

headers = {
    "Authorization": "Bearer XXX",
    "Content-Type": "application/json"
}

provincias = {
    "05": "Ávila",
    "09": "Burgos",
    "24": "León",
    "34": "Palencia",
    "37": "Salamanca",
    "40": "Segovia",
    "42": "Soria",
    "47": "Valladolid",
    "49": "Zamora"
}

ceas = []

for id_prov, nombre_prov in provincias.items():
    payload = {
        "catdiss": True,
        "idProvincia": id_prov,
        "activo": True,
        "numOrdenNotNull": True
    }

    try:
        response = rq.post(url, headers=headers, json=payload)
        response.raise_for_status()
        recursos = response.json()

        for r in recursos:
            nombre = r.get("nombre", "").strip()
            lat = r.get("latitud")
            lon = r.get("longitud")
        
            if nombre.upper().startswith("CEAS"):
                try:
                    lat = float(lat) if lat not in (None, "", "null") else np.nan
                    lon = float(lon) if lon not in (None, "", "null") else np.nan
                except ValueError:
                    lat, lon = np.nan, np.nan
        
                ceas.append({
                    "Nombre": nombre,
                    "Latitud": lat,
                    "Longitud": lon,
                    "Provincia": nombre_prov
                })
        
        print(f"{nombre_prov}: {len(recursos)} recursos consultados, {len([r for r in recursos if r['nombre'].startswith('CEAS')])} CEAS encontrados.")
    except Exception as e:
        print(f"Error en {nombre_prov}: {e}")

# Crear DataFrame
df_ceas = pd.DataFrame(ceas, columns=["Nombre", "Latitud", "Longitud", "Provincia"])

# Guardar CSV
ruta_salida = os.path.join(DATA_INPUTS_DA, "ceas.csv")
df_ceas.to_csv(ruta_salida, index=False, encoding="utf-8")

print(f"\n✅ Archivo 'ceas.csv' generado correctamente con {len(df_ceas)} registros.")

Ávila: 70 recursos consultados, 22 CEAS encontrados.
Burgos: 138 recursos consultados, 34 CEAS encontrados.
León: 132 recursos consultados, 42 CEAS encontrados.
Palencia: 55 recursos consultados, 16 CEAS encontrados.
Salamanca: 106 recursos consultados, 22 CEAS encontrados.
Segovia: 64 recursos consultados, 13 CEAS encontrados.
Soria: 31 recursos consultados, 12 CEAS encontrados.
Valladolid: 152 recursos consultados, 40 CEAS encontrados.
Zamora: 51 recursos consultados, 14 CEAS encontrados.

✅ Archivo 'ceas.csv' generado correctamente con 215 registros.
